# Create IRT groups for ASSISTments 2017

## Installs and imports

In [ ]:
# Install IRT package
!pip install -q py-irt==0.1.1

# Imports
import os
import random
import pandas as pd
import numpy as np
import torch
import pyro

from py_irt.models.one_param_logistic import OneParamLog

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
pyro.set_rng_seed(SEED)


device = "cpu"

from google.colab import drive
drive.mount("/content/drive")

# Paths
DATA_DIR = "/content/drive/MyDrive/education-ml-research/ASSISTments2017"

DATA_PATH = os.path.join(
    DATA_DIR,
    "anonymized_full_release_competition_dataset.csv"
)

ABILITY_GROUP_PATH = os.path.join(
    DATA_DIR,
    "student_irt_ability_groups.csv"
)

# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of students:", df["studentId"].nunique())
print("Number of skills:", df["skill"].nunique())
print("Number of problems:", df["problemId"].nunique())

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.0/258.0 kB 11.0 MB/s eta 0:00:00
Mounted at /content/drive


/tmp/ipykernel_1032/1323053064.py:41: DtypeWarning: Columns (76,77) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


Dataset shape: (942816, 82)
Number of students: 1709
Number of skills: 102
Number of problems: 3162

Columns:
['studentId', 'MiddleSchoolId', 'InferredGender', 'SY ASSISTments Usage', 'AveKnow', 'AveCarelessness', 'AveCorrect', 'NumActions', 'AveResBored', 'AveResEngcon', 'AveResConf', 'AveResFrust', 'AveResOfftask', 'AveResGaming', 'action_num', 'skill', 'problemId', 'problemType', 'assignmentId', 'assistmentId', 'startTime', 'endTime', 'timeTaken', 'correct', 'original', 'hint', 'hintCount', 'hintTotal', 'scaffold', 'bottomHint', 'attemptCount', 'frIsHelpRequest', 'frPast5HelpRequest', 'frPast8HelpRequest', 'stlHintUsed', 'past8BottomOut', 'totalFrPercentPastWrong', 'totalFrPastWrongCount', 'frPast5WrongCount', 'frPast8WrongCount', 'totalFrTimeOnSkill', 'timeSinceSkill', 'frWorkingInSchool', 'totalFrAttempted', 'totalFrSkillOpportunities', 'responseIsFillIn', 'responseIsChosen', 'endsWithScaffolding', 'endsWithAutoScaffolding', 'frTimeTakenOnScaffolding', 'frTotalSkillOpportunitiesSc

,studentId,MiddleSchoolId,InferredGender,SY ASSISTments Usage,AveKnow,AveCarelessness,AveCorrect,NumActions,AveResBored,AveResEngcon,...,RES_CONFUSED,RES_FRUSTRATED,RES_OFFTASK,RES_GAMING,Ln-1,Ln,MCAS,Enrolled,Selective,isSTEM
0,8,2,Male,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,...,0.000000,0.0,0.785585,0.000264,0.13,0.06119,45,0,0,NaN
1,8,2,Male,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,...,0.887452,0.0,0.468252,0.001483,0.06119,0.21351,45,0,0,NaN
2,8,2,Male,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,...,0.887452,0.0,0.468252,0.001483,0.116,0.033306,45,0,0,NaN
3,8,2,Male,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,...,0.000000,0.0,0.108417,0.010665,0.116,0.033306,45,0,0,NaN
4,8,2,Male,2004-2005,0.352416,0.183276,0.483902,1056,0.208389,0.679126,...,0.000000,0.0,0.108417,0.010665,0.033306,0.118386,45,0,0,NaN


## Prepare IRT data using all valid interactions

In [ ]:
irt_all = df[
    ["studentId", "problemId", "startTime", "correct"]
].copy()

irt_all = irt_all.dropna(
    subset=["studentId", "problemId", "startTime", "correct"]
)

irt_all["correct"] = pd.to_numeric(
    irt_all["correct"], errors="coerce"
)

irt_all = irt_all[
    irt_all["correct"].isin([0, 1])
].copy()

irt_all["startTime"] = pd.to_datetime(
    irt_all["startTime"], errors="coerce"
)

irt_all = irt_all.dropna(
    subset=["startTime"]
)

# Sort chronologically
irt_all = irt_all.sort_values(
    ["studentId", "startTime"],
    kind="stable"
).reset_index(drop=True)

# Robustness dataset: first recorded attempt per student-problem pair
irt_first = (
    irt_all
    .drop_duplicates(
        subset=["studentId", "problemId"],
        keep="first"
    )
    .reset_index(drop=True)
)

# Summary
print("ALL-INTERACTION DATASET")
print("Rows:", len(irt_all))
print("Students:", irt_all["studentId"].nunique())
print("Problems:", irt_all["problemId"].nunique())

print("\nFIRST-ATTEMPT DATASET")
print("Rows:", len(irt_first))
print("Students:", irt_first["studentId"].nunique())
print("Problems:", irt_first["problemId"].nunique())

print(
    "\nRepeated interactions removed:",
    len(irt_all) - len(irt_first)
)

print(
    "Percentage of interactions retained:",
    round(100 * len(irt_first) / len(irt_all), 2),
    "%"
)

ALL-INTERACTION DATASET
Rows: 942816
Students: 1709
Problems: 3162

FIRST-ATTEMPT DATASET
Rows: 390331
Students: 1709
Problems: 3162

Repeated interactions removed: 552485
Percentage of interactions retained: 41.4 %


## Convert student IDs, problem IDs, and responses into tensors

In [ ]:
def prepare_irt_data(data):

    student_ids = np.sort(
        data["studentId"].unique()
    )

    problem_ids = np.sort(
        data["problemId"].unique()
    )

    student_to_idx = {
        student_id: i
        for i, student_id in enumerate(student_ids)
    }

    problem_to_idx = {
        problem_id: i
        for i, problem_id in enumerate(problem_ids)
    }

    models = torch.tensor(
        data["studentId"].map(student_to_idx).values,
        dtype=torch.long
    )

    items = torch.tensor(
        data["problemId"].map(problem_to_idx).values,
        dtype=torch.long
    )

    responses = torch.tensor(
        data["correct"].values,
        dtype=torch.float32
    )

    return (
        models,
        items,
        responses,
        student_ids,
        problem_ids
    )


# Prepare all-interaction data
(
    models_all,
    items_all,
    responses_all,
    student_ids_all,
    problem_ids_all
) = prepare_irt_data(irt_all)


# Prepare first-attempt data
(
    models_first,
    items_first,
    responses_first,
    student_ids_first,
    problem_ids_first
) = prepare_irt_data(irt_first)


# Verify the tensors
print("ALL-INTERACTION TENSORS")
print("Students:", len(student_ids_all))
print("Problems:", len(problem_ids_all))
print("Response tensor shape:", responses_all.shape)

print("\nFIRST-ATTEMPT TENSORS")
print("Students:", len(student_ids_first))
print("Problems:", len(problem_ids_first))
print("Response tensor shape:", responses_first.shape)

print("\nResponse values in all-interaction data:")
print(torch.unique(responses_all))

ALL-INTERACTION TENSORS
Students: 1709
Problems: 3162
Response tensor shape: torch.Size([942816])

FIRST-ATTEMPT TENSORS
Students: 1709
Problems: 3162
Response tensor shape: torch.Size([390331])

Response values in all-interaction data:
tensor([0., 1.])


## Fit the all-interaction IRT model

In [ ]:
# Move tensors to the selected device
models_all = models_all.to(device)
items_all = items_all.to(device)
responses_all = responses_all.to(device)

# Clear parameters from any previous model
pyro.clear_param_store()

# Initialize the 1PL IRT model
irt_model_all = OneParamLog(
    priors="hierarchical",
    device=device,
    num_items=len(problem_ids_all),
    num_models=len(student_ids_all),
    verbose=True
)

# Train the model
NUM_EPOCHS = 1000

irt_model_all.fit(
    models=models_all,
    items=items_all,
    responses=responses_all,
    num_epochs=NUM_EPOCHS
)

[epoch 0001] loss: 14413276.2218
[epoch 0101] loss: 591626.6564
[epoch 0201] loss: 574004.9882
[epoch 0301] loss: 576312.1601
[epoch 0401] loss: 659154.0960
[epoch 0501] loss: 582094.2312
[epoch 0601] loss: 577588.2915
[epoch 0701] loss: 573594.5332
[epoch 0801] loss: 575238.9245
[epoch 0901] loss: 574608.0938
[epoch 1000] loss: 580590.6002


## Extract student ability estimates

In [ ]:
param_store = pyro.get_param_store()

theta_all = (
    param_store["loc_ability"]
    .detach()
    .cpu()
    .numpy()
)

# Associate each estimate with its student ID
irt_scores_all = pd.DataFrame({
    "studentId": student_ids_all,
    "theta_all": theta_all
})

# Inspect the estimates
print("Number of students:", len(irt_scores_all))
print("\nMissing estimates:", irt_scores_all["theta_all"].isna().sum())
print(
    "Infinite estimates:",
    np.isinf(irt_scores_all["theta_all"]).sum()
)

print("\nAbility summary:")
display(irt_scores_all["theta_all"].describe())

print("\nAbility quantiles:")
display(
    irt_scores_all["theta_all"].quantile(
        [0, 0.25, 0.5, 0.75, 1]
    )
)

Number of students: 1709

Missing estimates: 0
Infinite estimates: 0

Ability summary:


,theta_all
count,1709.000000
mean,-0.213705
std,0.603668
min,-3.266655
25%,-0.631283
50%,-0.312331
75%,0.116890
max,3.524201



Ability quantiles:


,theta_all
0.00,-3.266655
0.25,-0.631283
0.50,-0.312331
0.75,0.116890
1.00,3.524201


## Fit the first-interaction IRT model

In [ ]:
# Move first-interaction tensors to the selected device
models_first = models_first.to(device)
items_first = items_first.to(device)
responses_first = responses_first.to(device)

# Clear the all-interaction model's parameters
pyro.clear_param_store()

# Initialize the first-interaction IRT model
irt_model_first = OneParamLog(
    priors="hierarchical",
    device=device,
    num_items=len(problem_ids_first),
    num_models=len(student_ids_first),
    verbose=True
)

# Train the model
irt_model_first.fit(
    models=models_first,
    items=items_first,
    responses=responses_first,
    num_epochs=1000
)

[epoch 0001] loss: 2344152.3675
[epoch 0101] loss: 248291.6889
[epoch 0201] loss: 253176.6242
[epoch 0301] loss: 238555.2670
[epoch 0401] loss: 249630.8379
[epoch 0501] loss: 249156.4402
[epoch 0601] loss: 279514.2958
[epoch 0701] loss: 234281.5388
[epoch 0801] loss: 231036.5514
[epoch 0901] loss: 230500.1684
[epoch 1000] loss: 254392.9210


## Compare IRT ability estimates

In [ ]:
from scipy.stats import pearsonr, spearmanr

# Extract first-interaction ability estimates
param_store = pyro.get_param_store()

theta_first = (
    param_store["loc_ability"]
    .detach()
    .cpu()
    .numpy()
)

irt_scores_first = pd.DataFrame({
    "studentId": student_ids_first,
    "theta_first": theta_first
})

# Merge estimates for the same students
irt_comparison = irt_scores_all.merge(
    irt_scores_first,
    on="studentId",
    how="inner"
)

# Calculate correlations
pearson_r, pearson_p = pearsonr(
    irt_comparison["theta_all"],
    irt_comparison["theta_first"]
)

spearman_rho, spearman_p = spearmanr(
    irt_comparison["theta_all"],
    irt_comparison["theta_first"]
)

print("Students compared:", len(irt_comparison))
print(f"Pearson correlation:  r = {pearson_r:.4f}")
print(f"Pearson p-value:      p = {pearson_p:.6g}")
print(f"Spearman correlation: rho = {spearman_rho:.4f}")
print(f"Spearman p-value:     p = {spearman_p:.6g}")

print("\nFirst-interaction ability summary:")
display(irt_comparison["theta_first"].describe())

Students compared: 1709
Pearson correlation:  r = 0.9392
Pearson p-value:      p = 0
Spearman correlation: rho = 0.9617
Spearman p-value:     p = 0

First-interaction ability summary:


,theta_first
count,1709.000000
mean,-0.025999
std,0.917015
min,-4.591161
25%,-0.658219
50%,-0.055382
75%,0.583449
max,4.202458


## Use all-interaction IRT estimates as the primary ability measure

In [ ]:
ability = irt_scores_all.copy()

# 1. Assign students to four equal-sized ability quartiles
ability["theta_quartile"] = pd.qcut(
    ability["theta_all"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

# 2. Create standard-deviation bands centered on theta = 0
theta_sd = ability["theta_all"].std()

sd_multipliers = [
    -np.inf, -1.5, -1.0, -0.5,
    0, 0.5, 1.0, 1.5, np.inf
]

sd_edges = [m * theta_sd for m in sd_multipliers]

sd_labels = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

ability["theta_sd_group"] = pd.cut(
    ability["theta_all"],
    bins=sd_edges,
    labels=sd_labels,
    include_lowest=True,
    right=False
)

# Display group sizes
print("QUARTILE GROUP SIZES")
display(
    ability["theta_quartile"]
    .value_counts()
    .sort_index()
)

print("\nSTANDARD-DEVIATION GROUP SIZES")
display(
    ability["theta_sd_group"]
    .value_counts()
    .reindex(sd_labels)
)

print("\nTheta standard deviation:", theta_sd)
print("Theta = 0 is the reference point for SD bands.")

QUARTILE GROUP SIZES


,count
theta_quartile,
Q1,428
Q2,427
Q3,427
Q4,427



STANDARD-DEVIATION GROUP SIZES


,count
theta_sd_group,
< -1.5 SD,136
-1.5 to -1.0 SD,330
-1.0 to -0.5 SD,407
-0.5 to 0 SD,314
0 to +0.5 SD,232
+0.5 to +1.0 SD,134
+1.0 to +1.5 SD,82
> +1.5 SD,74



Theta standard deviation: 0.6036679148674011
Theta = 0 is the reference point for SD bands.


## Save the groups

In [ ]:
# Combine both IRT estimates with the primary ability groups
ability_final = ability.merge(
    irt_scores_first[["studentId", "theta_first"]],
    on="studentId",
    how="left"
)

# Select and order the columns
ability_final = ability_final[
    [
        "studentId",
        "theta_all",
        "theta_first",
        "theta_quartile",
        "theta_sd_group"
    ]
]

# Save the file
ability_final.to_csv(
    ABILITY_GROUP_PATH,
    index=False
)

print("Saved to:", ABILITY_GROUP_PATH)
print("Rows:", len(ability_final))
print("\nMissing values:")
print(ability_final.isna().sum())

print("\nPreview:")
display(ability_final.head())

Saved to: /content/drive/MyDrive/education-ml-research/ASSISTments2017/student_irt_ability_groups.csv
Rows: 1709

Missing values:
studentId         0
theta_all         0
theta_first       0
theta_quartile    0
theta_sd_group    0
dtype: int64

Preview:


,studentId,theta_all,theta_first,theta_quartile,theta_sd_group
0,8,0.145408,0.659446,Q4,0 to +0.5 SD
1,9,0.040027,0.334709,Q3,0 to +0.5 SD
2,11,-0.626556,-0.538717,Q2,-1.5 to -1.0 SD
3,25,0.339209,0.544214,Q4,+0.5 to +1.0 SD
4,27,-0.264493,0.360488,Q3,-0.5 to 0 SD
